In [8]:
import pandas as pd
import os
from pathlib import Path
import pprint
import csv

In [15]:
def read_metadata(file_path):
    with open(file_path, 'r') as f:
        # Create a generator that filters out lines starting with #
        filtered_file = (line for line in f if line.startswith('#'))

        # Pass that generator to the DictReader
        reader = csv.DictReader(filtered_file, delimiter='\t')

        data = list(reader)
        # Display in a readable format
        pprint.pprint(data)

# Transcriptional


In [2]:
def consolidate_gdc_data(sample_sheet_path, data_dir, output_path, data_type_filter):
    """
    Consolidates individual GDC files into a single Parquet file.

    Args:
        sample_sheet_path (str): Absolute path to the GDC sample sheet.
        data_dir (str): Absolute path to the directory containing the 430 folders/files.
        output_path (str): Where to save the resulting Parquet file.
        data_type_filter (str): 'Gene Expression Quantification' or 'miRNA Expression Quantification'
    """
    # 1. Load the Sample Sheet
    sheet = pd.read_csv(sample_sheet_path, sep='\t')

    # Filter for the specific data type (RNA-seq or miRNA-seq)
    tumor_samples = sheet[sheet['Tissue Type'].str.contains('Tumor', case=False)].copy()
    filtered_sheet = tumor_samples[tumor_samples['Data Type'] == data_type_filter].copy()

    all_data = []

    print(f"Starting processing for {data_type_filter}...")

    for index, row in filtered_sheet.iterrows():
        file_id = row['File ID']
        file_name = row['File Name']

        file_path = os.path.join(data_dir, file_id, file_name)

        if not os.path.exists(file_path):
            print(f"Warning: File not found: {file_path}")
            continue

        # 2. Read the specific file
        if "miRNA" in data_type_filter:
            df = pd.read_csv(file_path, sep='\t')
            id_col = df.columns[0]
            val_col = 'reads_per_million_miRNA_mapped'
        else:
            # RNA-seq files: skip metadata lines
            df = pd.read_csv(file_path, sep='\t', comment='#')
            id_col = 'gene_id'
            val_col = 'tpm_unstranded'

        # The choice of "value" can be further investigated by looking at there distribution in crosspondence to the used model

        # Pivot the data to make genes the columns
        temp_df = df[[id_col, val_col]].set_index(id_col).T

        # 3. Inject Metadata
        temp_df.insert(0, 'patient_id', row['Case ID'])
        temp_df.insert(1, 'sample_id', row['Sample ID'])

        all_data.append(temp_df)

    # 4. Concatenate and Export
    final_df = pd.concat(all_data, axis=0).reset_index(drop=True)
    final_df.to_parquet(output_path, engine='pyarrow')
    print(f"Success! Saved {len(final_df)} samples to {output_path}")



In [13]:
# --- EXECUTION ---
SAMPLE_SHEET = "/media/maro/Mom0-0/Datasets/TCGA-STAD-transcriptional/gdc_sample_sheet.2026-04-17.tsv"
RNA_DIR = "/media/maro/Mom0-0/Datasets/TCGA-STAD-transcriptional"
MIRNA_DIR = "/media/maro/Mom0-0/Datasets/TCGA-STAD-transcriptional"



In [4]:
# Process RNA-Seq
consolidate_gdc_data(
    sample_sheet_path=SAMPLE_SHEET,
    data_dir=RNA_DIR,
    output_path="../../../data/raw/stad_rna_consolidated.parquet",
    data_type_filter="Gene Expression Quantification"
)


Starting processing for Gene Expression Quantification...
Success! Saved 412 samples to ../../../data/processed/stad_rna_consolidated.parquet


In [5]:
# Process miRNA-Seq
consolidate_gdc_data(
    sample_sheet_path=SAMPLE_SHEET,
    data_dir=MIRNA_DIR,
    output_path="../../../data/raw/stad_mirna_consolidated.parquet",
    data_type_filter="miRNA Expression Quantification"
)

Starting processing for miRNA Expression Quantification...
Success! Saved 446 samples to ../../../data/processed/stad_mirna_consolidated.parquet


In [22]:
read_metadata(SAMPLE_SHEET)

[]


# Clinical Patient

In [1]:
clinical_patient_path = "/media/maro/Mom0-0/Datasets/TMB Prediction/downloads_valid/stad_tcga_pan_can_atlas_2018/data_clinical_patient.txt"

In [5]:
clinical_patient_df = pd.read_csv(clinical_patient_path, sep='\t', comment='#')
clinical_patient_df.columns

Index(['PATIENT_ID', 'SUBTYPE', 'CANCER_TYPE_ACRONYM', 'OTHER_PATIENT_ID',
       'AGE', 'SEX', 'AJCC_PATHOLOGIC_TUMOR_STAGE', 'AJCC_STAGING_EDITION',
       'DAYS_LAST_FOLLOWUP', 'DAYS_TO_BIRTH',
       'DAYS_TO_INITIAL_PATHOLOGIC_DIAGNOSIS', 'ETHNICITY',
       'FORM_COMPLETION_DATE', 'HISTORY_NEOADJUVANT_TRTYN', 'ICD_10',
       'ICD_O_3_HISTOLOGY', 'ICD_O_3_SITE', 'INFORMED_CONSENT_VERIFIED',
       'NEW_TUMOR_EVENT_AFTER_INITIAL_TREATMENT', 'PATH_M_STAGE',
       'PATH_N_STAGE', 'PATH_T_STAGE', 'PERSON_NEOPLASM_CANCER_STATUS',
       'PRIMARY_LYMPH_NODE_PRESENTATION_ASSESSMENT', 'PRIOR_DX', 'RACE',
       'RADIATION_THERAPY', 'WEIGHT', 'IN_PANCANPATHWAYS_FREEZE', 'OS_STATUS',
       'OS_MONTHS', 'DSS_STATUS', 'DSS_MONTHS', 'DFS_STATUS', 'DFS_MONTHS',
       'PFS_STATUS', 'PFS_MONTHS', 'GENETIC_ANCESTRY_LABEL'],
      dtype='str')

In [10]:
read_metadata(clinical_patient_path)

[{'#Patient Identifier': '#Identifier to uniquely specify a patient.',
  'American Joint Committee on Cancer Metastasis Stage Code': 'Code to '
                                                              'represent the '
                                                              'defined absence '
                                                              'or presence of '
                                                              'distant spread '
                                                              'or metastases '
                                                              '(M) to '
                                                              'locations via '
                                                              'vascular '
                                                              'channels or '
                                                              'lymphatics '
                                                              'beyond the '
   

In [23]:
clinical_patient_df.drop(columns=['CANCER_TYPE_ACRONYM', 'OTHER_PATIENT_ID'], inplace=True)

In [24]:
clinical_patient_df.to_parquet("../../../data/raw/stad_clinical_patient.parquet", engine='pyarrow')

# TMB Target

In [27]:
clinical_sample_path = "/media/maro/Mom0-0/Datasets/TMB Prediction/downloads_valid/stad_tcga_pan_can_atlas_2018/data_clinical_sample.txt"


In [28]:
clinical_sample_df = pd.read_csv(clinical_sample_path, sep='\t', comment='#')
clinical_sample_df.columns

Index(['PATIENT_ID', 'SAMPLE_ID', 'ONCOTREE_CODE', 'CANCER_TYPE',
       'CANCER_TYPE_DETAILED', 'TUMOR_TYPE', 'GRADE',
       'TISSUE_PROSPECTIVE_COLLECTION_INDICATOR',
       'TISSUE_RETROSPECTIVE_COLLECTION_INDICATOR', 'TISSUE_SOURCE_SITE_CODE',
       'TUMOR_TISSUE_SITE', 'ANEUPLOIDY_SCORE', 'SAMPLE_TYPE',
       'MSI_SCORE_MANTIS', 'MSI_SENSOR_SCORE', 'SOMATIC_STATUS',
       'TMB_NONSYNONYMOUS', 'TISSUE_SOURCE_SITE', 'TBL_SCORE'],
      dtype='str')

In [29]:
clinical_sample_df[['PATIENT_ID','SAMPLE_ID','TMB_NONSYNONYMOUS']].to_parquet("../../../data/raw/stad_tmb.parquet", engine='pyarrow')
